# Customer Churn Prediction Using Machine Learning

Machine Learning for Business Applications | MBA - Business Analytics

## 1. Business Problem
A telecom company wants to identify customers likely to leave so retention actions can be taken before churn occurs.

**Target:** Churn

**Business decision:** Prioritize high-risk customers for retention campaigns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import joblib

In [ ]:
df = pd.read_csv('../data/Telco-Customer-Churn.csv')
df.head()

In [ ]:
df.info()
df.shape

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.isnull().sum()
df = df.dropna().copy()

In [ ]:
df['ChurnFlag'] = (df['Churn'] == 'Yes').astype(int)
df['Churn'].value_counts()

## 2. Exploratory Data Analysis

In [ ]:
df['Churn'].value_counts().plot(kind='bar')
plt.title('Customer Churn Distribution'); plt.xlabel('Churn'); plt.ylabel('Customers'); plt.show()

In [ ]:
df.groupby('Contract')['ChurnFlag'].mean().sort_values().plot(kind='bar')
plt.title('Churn Rate by Contract'); plt.ylabel('Churn Rate'); plt.show()

In [ ]:
df.groupby('InternetService')['ChurnFlag'].mean().sort_values().plot(kind='bar')
plt.title('Churn Rate by Internet Service'); plt.ylabel('Churn Rate'); plt.show()

In [ ]:
df.groupby('SeniorCitizen')['ChurnFlag'].mean().plot(kind='bar')
plt.title('Churn Rate by Senior Citizen Status'); plt.ylabel('Churn Rate'); plt.show()

In [ ]:
df.groupby('tenure')['ChurnFlag'].mean().rolling(6,min_periods=1).mean().plot()
plt.title('Smoothed Churn Rate by Tenure'); plt.ylabel('Churn Rate'); plt.xlabel('Tenure'); plt.show()

## 3. Model Development

In [ ]:
X = df.drop(columns=['Churn','ChurnFlag','customerID'])
y = df['ChurnFlag']
num_cols = X.select_dtypes(exclude='object').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

In [ ]:
pre = ColumnTransformer([('num',StandardScaler(),num_cols),('cat',OneHotEncoder(handle_unknown='ignore',drop='first'),cat_cols)])
log_model = Pipeline([('preprocess',pre),('model',LogisticRegression(max_iter=2000))])
log_model.fit(X_train,y_train)
log_pred=log_model.predict(X_test)
log_prob=log_model.predict_proba(X_test)[:,1]

In [ ]:
pre_rf = ColumnTransformer([('num','passthrough',num_cols),('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)])
rf_model=Pipeline([('preprocess',pre_rf),('model',RandomForestClassifier(n_estimators=300,random_state=42,class_weight='balanced',n_jobs=-1))])
rf_model.fit(X_train,y_train)
rf_pred=rf_model.predict(X_test)
rf_prob=rf_model.predict_proba(X_test)[:,1]

## 4. Model Evaluation

In [ ]:
def evaluate(y_true, pred, prob):
    return {
        'Accuracy': accuracy_score(y_true,pred),
        'Precision': precision_score(y_true,pred),
        'Recall': recall_score(y_true,pred),
        'F1': f1_score(y_true,pred),
        'ROC-AUC': roc_auc_score(y_true,prob)
    }
results = pd.DataFrame({'Logistic Regression':evaluate(y_test,log_pred,log_prob),
                        'Random Forest':evaluate(y_test,rf_pred,rf_prob)})
results

In [ ]:
cm=confusion_matrix(y_test,log_pred)
plt.imshow(cm); plt.title('Logistic Regression Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.colorbar(); plt.show()

## 5. Business Interpretation
Logistic Regression is selected because it gives stronger overall test performance than Random Forest. The model's probability output can be used to prioritize customers for retention. It should be treated as decision support rather than a guarantee that a customer will churn.

In [ ]:
joblib.dump(log_model,'../model.pkl')